# Scope-3 Emissions Snapshot & Supplier Risk Score

This notebook implements:
- Delivered-only filtering
- Defensive EF column selection (with/without margins)
- Imputation by NAICS median and flagging
- Safe min-max normalization
- Composite Priority Score calculation
- Emissions reported in tCO2e


## 1. Load Datasets

In [ ]:
import pandas as pd
import numpy as np
import os

# -----------------------------------------------------------
# PATHS: place your datasets in the data/ folder
# -----------------------------------------------------------
proc_path = 'data/procurement_dataset.csv'
em_path   = 'data/SupplyChainGHGEmissionFactors_v1.2_NAICS_CO2e_USD_2021.csv'

print(os.path.exists(proc_path), os.path.exists(em_path))

proc_df = pd.read_csv(proc_path)
em_df   = pd.read_csv(em_path)

print('proc rows', len(proc_df), 'em rows', len(em_df))
proc_df.head()

## 2. Emission Factor Column Selection

In [ ]:
print('EF columns', em_df.columns.tolist())

ef_candidates = ['Supply Chain Emission Factors with Margins',
                 'Supply Chain Emission Factors without Margins']
selected = None
for c in ef_candidates:
    if c in em_df.columns:
        selected = c
        break
if not selected:
    raise ValueError('No EF column found')
print('Selected EF column:', selected)

naics_candidates = ['2017 NAICS Code', '2012 NAICS Code']
ncol = None
for c in naics_candidates:
    if c in em_df.columns:
        ncol = c
        break
if ncol is None:
    raise ValueError('No NAICS column found')
print('Selected NAICS column:', ncol)

em = em_df.rename(columns={ncol: 'NAICS6', selected: 'EF_kg_per_USD'})
em['NAICS6'] = em['NAICS6'].astype(str).str.split('.').str[0].str.strip()
em['EF_kg_per_USD'] = pd.to_numeric(em['EF_kg_per_USD'], errors='coerce')
em[['NAICS6','EF_kg_per_USD']].head()

## 3. Category → NAICS Mapping

In [ ]:
manual_map = {
    'Office Supplies': '453210',
    'Electronics':     '334111',
    'MRO':             '333999',
    'Packaging':       '322211',
    'Raw Materials':   '325199'
}

catmap = pd.DataFrame(list(manual_map.items()), columns=['Item_Category','NAICS6'])
os.makedirs('data', exist_ok=True)
catmap.to_csv('data/category_to_naics.csv', index=False)
print('Saved mapping to data/category_to_naics.csv')
catmap

## 4. Data Cleaning & Spend Calculation

In [ ]:
proc = proc_df.copy()
proc['Order_Date']    = pd.to_datetime(proc['Order_Date'],    errors='coerce')
proc['Delivery_Date'] = pd.to_datetime(proc['Delivery_Date'], errors='coerce')

valid_statuses = ['Delivered', 'Partially Delivered']
if 'Order_Status' in proc.columns:
    proc = proc[proc['Order_Status'].isin(valid_statuses)].copy()

proc['Negotiated_Price'] = pd.to_numeric(proc.get('Negotiated_Price'), errors='coerce')
proc['Unit_Price']       = pd.to_numeric(proc.get('Unit_Price'),       errors='coerce')
proc['Quantity']         = pd.to_numeric(proc.get('Quantity'),         errors='coerce').fillna(0)

proc['Effective_Price'] = proc['Negotiated_Price'].fillna(proc['Unit_Price'].fillna(0))
proc = proc[(proc['Quantity'] > 0) & (proc['Effective_Price'] > 0)].copy()

proc['Spend_USD'] = proc['Quantity'] * proc['Effective_Price']
proc['Period']    = proc['Order_Date'].dt.year

print('Valid lines:', len(proc))
proc.head()

## 5. Join, Imputation & Emissions Calculation

In [ ]:
proc = proc.merge(catmap, on='Item_Category', how='left')
proc['NAICS6'] = proc['NAICS6'].fillna('UNKNOWN')

merged = proc.merge(em[['NAICS6','EF_kg_per_USD']], on='NAICS6', how='left')

# Track which EFs were originally missing
ef_missing_initial = merged['EF_kg_per_USD'].isna()

# Impute via NAICS median
median_by_naics = em.groupby('NAICS6')['EF_kg_per_USD'].median()
merged['EF_imputed'] = False
merged['EF_kg_per_USD'] = merged['EF_kg_per_USD'].fillna(merged['NAICS6'].map(median_by_naics))
merged.loc[ef_missing_initial & merged['EF_kg_per_USD'].notna(), 'EF_imputed'] = True
merged['EF_kg_per_USD'] = merged['EF_kg_per_USD'].fillna(0)

# Scope-3 emissions: spend-based methodology
merged['Emissions_tCO2e'] = merged['Spend_USD'] * merged['EF_kg_per_USD'] / 1000.0

merged[['PO_ID','Supplier','Item_Category','NAICS6','Spend_USD',
        'EF_kg_per_USD','EF_imputed','Emissions_tCO2e']].head()

## 6. Supplier Aggregation & Priority Score

In [ ]:
def safe_normalize(s):
    s = pd.Series(s).astype(float).fillna(0)
    rng = s.max() - s.min()
    if rng == 0:
        return pd.Series(0, index=s.index)
    return (s - s.min()) / rng

sup = merged.groupby(['Supplier','Period'], as_index=False).agg(
    Total_Spend_USD       =('Spend_USD',        'sum'),
    Total_Emissions_tCO2e =('Emissions_tCO2e',  'sum'),
    PO_Lines              =('PO_ID',             'count'),
    Imputed_Fraction      =('EF_imputed',        'mean'),
)

sup['Spend_Norm']      = safe_normalize(sup['Total_Spend_USD'])
sup['Emissions_Norm']  = safe_normalize(sup['Total_Emissions_tCO2e'])
sup['Gap_Norm']        = safe_normalize(sup['Imputed_Fraction'])
sup['Priority_Score']  = 0.5*sup['Emissions_Norm'] + 0.5*sup['Spend_Norm'] + 0.0*sup['Gap_Norm']

sup = sup.sort_values(['Period','Priority_Score'], ascending=[True, False]).reset_index(drop=True)
sup.head(10)

## 7. Save Outputs

In [ ]:
out_dir = 'data'
os.makedirs(out_dir, exist_ok=True)

merged.to_csv(f'{out_dir}/line_level_emissions_with_imputation.csv', index=False)
sup.to_csv(f'{out_dir}/supplier_summary_with_priority.csv', index=False)
sup.groupby('Period').head(20).to_csv(f'{out_dir}/top20_suppliers_per_period.csv', index=False)

print('Saved outputs to', out_dir)

## 8. Visualisation — Top 10 Suppliers by Emissions

In [ ]:
import matplotlib.pyplot as plt

top10 = sup.nlargest(10, 'Total_Emissions_tCO2e')

plt.figure(figsize=(10, 6))
plt.barh(top10['Supplier'][::-1], top10['Total_Emissions_tCO2e'][::-1], color='steelblue')
plt.xlabel('Estimated Emissions (tCO2e)')
plt.title('Top 10 Suppliers by Estimated Scope-3 Emissions (tCO2e)')
plt.tight_layout()
plt.savefig('data/top10_emissions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Results Summary

In [ ]:
print('Total valid procurement lines:', len(merged))
print('Total spend USD:', merged['Spend_USD'].sum())
print('Total emissions tCO2e:', merged['Emissions_tCO2e'].sum())
print('Unique suppliers:', merged['Supplier'].nunique())
print('Coverage after imputation :', 100 * (merged['EF_kg_per_USD'] > 0).sum() / len(merged))
print('\nTop 5 suppliers by emissions:')
print(sup.nlargest(5, 'Total_Emissions_tCO2e')[['Supplier','Total_Emissions_tCO2e','Total_Spend_USD','Priority_Score']])